# Python für Verfahrenstechniker — Teil 4
## Messdaten, gute Praxis und Abschlussprojekt: Batch-Reaktor

Im letzten Teil bringen Sie alles zusammen. Sie lesen echte Messdaten aus einer Datei ein,
werten sie aus, bestimmen daraus eine Reaktionsgeschwindigkeitskonstante und vergleichen Modell
und Experiment — also genau der Ablauf eines Praktikumsprotokolls.

---

### Lernziele

Nach diesem Notebook können Sie

- Messdaten aus einer CSV-Datei einlesen und auswerten,
- eine Ausgleichsgerade durch linearisierte Daten legen,
- ein vollständiges kleines Auswerteprogramm strukturieren,
- benennen, was guten von schlechtem Code unterscheidet,
- einschätzen, wo die Grenze zwischen analytischer Lösung und numerischem Verfahren verläuft.

**Voraussetzung:** Teile 1 bis 3. **Zeitaufwand:** ca. 120 Minuten.

> **Benötigte Datei:** `messdaten_batchreaktor.csv` muss im selben Ordner wie dieses Notebook liegen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Messdaten einlesen

Messdaten kommen selten als getippte Python-Liste, sondern als Textdatei aus einem Messsystem
oder als Export aus Excel. Sehen wir uns zuerst an, was in der Datei steht:

In [ ]:
with open("messdaten_batchreaktor.csv", encoding="utf-8") as datei:
    for i, zeile in enumerate(datei):
        print(zeile.rstrip())
        if i > 9:
            print("...")
            break

Typischer Aufbau: einige Kommentarzeilen mit `#`, dann eine Kopfzeile mit den Spaltennamen,
dann die Zahlen — hier durch Kommata getrennt.

**Sehen Sie sich eine Datei immer erst an, bevor Sie sie einlesen.** Wie viele Kopfzeilen? Welches
Trennzeichen? Punkt oder Komma als Dezimaltrenner? Fehlende Werte? Diese vier Fragen ersparen
Ihnen die Hälfte aller Einleseprobleme.

In [ ]:
daten = np.loadtxt(
    "messdaten_batchreaktor.csv",
    delimiter=",",     # Spalten sind durch Komma getrennt
    comments="#",      # Zeilen mit # werden übersprungen
    skiprows=6,        # 5 Kommentarzeilen + 1 Kopfzeile
)

print("Form der Daten:", daten.shape)     # (Zeilen, Spalten)
print(daten[:5])                          # die ersten fünf Zeilen

Das Ergebnis ist ein zweidimensionales Array. Die Spalten holen Sie sich einzeln heraus und geben
ihnen sofort **sprechende Namen mit Einheit im Kommentar** — arbeiten Sie nie länger als nötig
mit `daten[:, 0]`.

In [ ]:
t   = daten[:, 0]     # Zeit in min          (alle Zeilen, Spalte 0)
c_A = daten[:, 1]     # Konzentration in mol/L

print(f"{t.size} Messpunkte von {t.min():.0f} bis {t.max():.0f} min")
print(f"Konzentration von {c_A.max():.3f} auf {c_A.min():.3f} mol/L gefallen")

### Erster Blick: immer zuerst zeichnen

Bevor Sie irgendetwas rechnen, sehen Sie sich die Daten an. Ausreißer, Sättigungseffekte oder ein
falsch eingestelltes Messgerät erkennen Sie im Diagramm sofort — in der Zahlenkolonne nicht.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(t, c_A, "o")
plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"Konzentration $c_\mathrm{A}$ / mol L$^{-1}$")
plt.title("Batch-Reaktor: Konzentrationsverlauf, Rohdaten")
plt.grid(True)
plt.show()

Der Verlauf sieht nach einem exponentiellen Abfall aus — verträglich mit einer Reaktion erster
Ordnung. Bei kleinen Konzentrationen streuen die Werte relativ stärker, was für eine
konzentrationsunabhängige Nachweisgrenze der Analytik spricht. Beides sind Beobachtungen, die
in ein Protokoll gehören.

---
## 2. Gute Praxis

Bevor wir zum Projekt kommen, ein kurzer Abschnitt darüber, was Ihren Code in einem halben Jahr
noch lesbar macht — für Ihre Betreuerin, für die Bewertung Ihrer Arbeit und vor allem für Sie selbst.

**1. Sprechende Namen statt Kürzel.**
`c_A_ein` statt `x1`. Der Tippaufwand ist einmalig, das Rätselraten dauert ewig.

**2. Einheiten dokumentieren.**
Python rechnet einheitenlos. Jede physikalische Größe bekommt einen Kommentar mit der Einheit.
Legen Sie sich außerdem an, konsequent in SI-Einheiten zu rechnen und nur bei der Ausgabe umzurechnen.

**3. Keine Zahlen im Code verstreuen.**
Konstanten gehören an den Anfang, mit Namen versehen:

```python
# schlecht
umsatz = 1 - c / 2.0

# gut
c_A0 = 2.0                # mol/L, Anfangskonzentration laut Ansatz
umsatz = 1 - c / c_A0
```

Ändert sich der Ansatz, ändern Sie eine Zeile statt sieben.

**4. Wiederholung in Funktionen packen.**
Sobald Sie eine Rechnung ein zweites Mal brauchen, wird sie zur Funktion. Zwei Kopien einer
Formel im Code bedeuten früher oder später zwei unterschiedliche Formeln.

**5. Kommentieren, warum — nicht was.**

```python
c_A = c_A[t <= 15]     # schlecht: "Werte auswählen"
c_A = c_A[t <= 15]     # gut: ab 15 min unter der Nachweisgrenze der HPLC
```

**6. Plausibilität prüfen.**
Massenbilanz schließt? Größenordnung stimmt? Grenzfall bekannt? Ein Ergebnis, das Sie nicht
geprüft haben, ist kein Ergebnis.

**7. Reproduzierbarkeit sicherstellen.**
Vor dem Abgeben *Kernel → Restart Kernel and Run All Cells*. Läuft das Notebook nicht von oben
durch, ist es nicht fertig.

---
## 3. Abschlussprojekt: Batch-Reaktor

### Aufgabenstellung

In einem diskontinuierlich betriebenen Rührkesselreaktor (Batch-Reaktor) läuft bei 60 °C die
Reaktion

$$\mathrm{A} \longrightarrow \mathrm{B}$$

ab. Bei ideal durchmischtem Inhalt und konstantem Volumen lautet die Stoffbilanz für A

$$\frac{\mathrm{d}c_\mathrm{A}}{\mathrm{d}t} = -r = -k \, c_\mathrm{A}$$

wenn die Reaktion erster Ordnung ist. Diese Differentialgleichung hat die bekannte Lösung

$$c_\mathrm{A}(t) = c_{\mathrm{A}0} \, \mathrm{e}^{-k t}$$

**Ihre Aufgaben:**

1. Prüfen Sie grafisch, ob die Daten wirklich zu einer Reaktion erster Ordnung passen.
2. Bestimmen Sie die Geschwindigkeitskonstante *k* aus den Messdaten.
3. Vergleichen Sie das Modell mit den Messwerten.
4. Berechnen Sie Umsatz und Halbwertszeit.
5. Sagen Sie den Verlauf bei 80 °C voraus (Arrhenius).

Arbeiten Sie die Schritte 3.1 bis 3.5 nacheinander durch.

### 3.1 Linearisierung — ist die Reaktion erster Ordnung?

Logarithmiert man die Lösung, erhält man eine Geradengleichung:

$$\ln c_\mathrm{A} = \ln c_{\mathrm{A}0} - k\,t$$

Trägt man also $\ln c_\mathrm{A}$ über $t$ auf, muss sich für eine Reaktion erster Ordnung eine
**Gerade** ergeben, deren Steigung $-k$ ist. Krümmt sich die Auftragung erkennbar, ist die
Reaktionsordnung eine andere.

In [ ]:
ln_c = np.log(c_A)

plt.figure(figsize=(7, 4.5))
plt.plot(t, ln_c, "o")
plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"$\ln(c_\mathrm{A}\,/\,\mathrm{mol\,L^{-1}})$")
plt.title("Linearisierte Auftragung (1. Ordnung)")
plt.grid(True)
plt.show()

Die Punkte liegen bis auf die Streuung am Ende gut auf einer Geraden. Die Annahme einer Reaktion
erster Ordnung ist damit gerechtfertigt.

### 3.2 Ausgleichsgerade und Geschwindigkeitskonstante

Die Gerade legen Sie mit `np.polyfit` durch die Punkte. Das Verfahren dahinter ist die Methode der
kleinsten Fehlerquadrate; `deg=1` bedeutet ein Polynom ersten Grades.

In [ ]:
steigung, achsenabschnitt = np.polyfit(t, ln_c, deg=1)

c_A0_ansatz = 2.0                      # mol/L, laut Rezeptur eingewogen

k = -steigung                          # 1/min
c_A0_fit = np.exp(achsenabschnitt)     # mol/L

print(f"Steigung:               {steigung:.5f} 1/min")
print(f"Geschwindigkeitskonstante:  k = {k:.4f} 1/min")
print(f"Anfangskonzentration aus Fit:   c_A0 = {c_A0_fit:.3f} mol/L")
print(f"Anfangskonzentration laut Ansatz: c_A0 = {c_A0_ansatz:.3f} mol/L")

Der aus der Regression bestimmte Achsenabschnitt liefert eine Anfangskonzentration nahe am
Ansatzwert von 2,0 mol/L. Das ist eine **unabhängige Kontrolle**: Weicht dieser Wert deutlich ab,
stimmt etwas mit dem Ansatz, der Analytik oder dem Modell nicht.

### Bestimmtheitsmaß

Wie gut die Gerade passt, quantifiziert das Bestimmtheitsmaß $R^2$:

In [ ]:
ln_c_fit = steigung * t + achsenabschnitt

ss_res = np.sum((ln_c - ln_c_fit) ** 2)              # Restquadratsumme
ss_tot = np.sum((ln_c - ln_c.mean()) ** 2)           # Gesamtquadratsumme
R2 = 1 - ss_res / ss_tot

print(f"R^2 = {R2:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(t, ln_c, "o", label="Messwerte")
plt.plot(t, ln_c_fit, "-", label=f"Ausgleichsgerade, $k$ = {k:.4f} min$^{{-1}}$")

plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"$\ln(c_\mathrm{A}\,/\,\mathrm{mol\,L^{-1}})$")
plt.title(f"Regression, $R^2$ = {R2:.4f}")
plt.legend()
plt.grid(True)
plt.show()

### 3.3 Modell und Messung vergleichen

Jetzt zurück in die ursprüngliche Darstellung: berechnete Kurve als Linie, Messwerte als Symbole.

In [ ]:
def konzentration_erster_ordnung(t, c_A0, k):
    """Konzentration einer Reaktion 1. Ordnung im Batch-Reaktor.

    t    : Zeit in min (Zahl oder Array)
    c_A0 : Anfangskonzentration in mol/L
    k    : Geschwindigkeitskonstante in 1/min

    Rückgabe: Konzentration in mol/L
    """
    return c_A0 * np.exp(-k * t)


t_fein = np.linspace(0, 20, 200)
c_modell = konzentration_erster_ordnung(t_fein, c_A0_fit, k)

plt.figure(figsize=(7, 4.5))
plt.plot(t_fein, c_modell, "-", label="Modell 1. Ordnung")
plt.plot(t, c_A, "o", label="Messwerte")

plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"Konzentration $c_\mathrm{A}$ / mol L$^{-1}$")
plt.title("Batch-Reaktor: Modell und Messung")
plt.legend()
plt.grid(True)
plt.show()

### Residuen ansehen

Die Abweichungen zwischen Messung und Modell — die **Residuen** — sollten zufällig um null streuen.
Ein systematisches Muster (etwa erst alle Punkte über, dann alle unter der Kurve) wäre ein Hinweis,
dass das Modell die Physik nicht richtig trifft.

In [ ]:
residuen = c_A - konzentration_erster_ordnung(t, c_A0_fit, k)

plt.figure(figsize=(7, 3.5))
plt.plot(t, residuen, "o")
plt.axhline(0, color="gray", linewidth=1)
plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"$c_\mathrm{A,mess} - c_\mathrm{A,modell}$ / mol L$^{-1}$")
plt.title("Residuen")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"mittlere Abweichung: {np.abs(residuen).mean():.4f} mol/L")
print(f"maximale Abweichung: {np.abs(residuen).max():.4f} mol/L")

### 3.4 Umsatz und Halbwertszeit

Der Umsatz ist definiert als

$$X = \frac{c_{\mathrm{A}0} - c_\mathrm{A}}{c_{\mathrm{A}0}} = 1 - \mathrm{e}^{-kt}$$

Nach dem Modell ergibt sich die Zeit für einen geforderten Umsatz zu

$$t = -\frac{\ln(1 - X)}{k}$$

In [ ]:
def zeit_fuer_umsatz(X, k):
    """Erforderliche Reaktionszeit in min für Umsatz X bei Reaktion 1. Ordnung."""
    return -np.log(1 - X) / k


for X in [0.50, 0.90, 0.95, 0.99]:
    t_erf = zeit_fuer_umsatz(X, k)
    print(f"Umsatz {X*100:5.1f} %  ->  t = {t_erf:6.2f} min")

Der Wert für 50 % Umsatz ist die **Halbwertszeit** $t_{1/2} = \ln 2 / k$. Beachten Sie die
Konsequenz für die Auslegung: Von 90 % auf 99 % Umsatz verdoppelt sich die benötigte Reaktionszeit
nahezu. Die letzten Prozent Umsatz sind immer die teuersten.

In [ ]:
X_verlauf = 1 - np.exp(-k * t_fein)

plt.figure(figsize=(7, 4.5))
plt.plot(t_fein, X_verlauf * 100)

for X in [0.50, 0.90, 0.99]:
    t_erf = zeit_fuer_umsatz(X, k)
    plt.plot([0, t_erf, t_erf], [X*100, X*100, 0], "--", color="gray", linewidth=1)

plt.xlabel("Zeit $t$ / min")
plt.ylabel("Umsatz $X$ / %")
plt.title("Umsatzverlauf")
plt.xlim(0, 20)
plt.ylim(0, 100)
plt.grid(True)
plt.show()

### 3.5 Temperaturabhängigkeit: Arrhenius

Die Geschwindigkeitskonstante hängt nach Arrhenius von der Temperatur ab:

$$k(T) = k_0 \, \exp\!\left(-\frac{E_A}{R\,T}\right)$$

Aus einem einzigen Versuch lässt sich $E_A$ nicht bestimmen — dafür bräuchten Sie Messungen bei
mehreren Temperaturen. Nehmen wir $E_A = 75$ kJ/mol aus der Literatur an. Dann können Sie den
Stoßfaktor $k_0$ aus Ihrem Messwert bei 60 °C bestimmen und auf andere Temperaturen umrechnen.

In [ ]:
R = 8.314           # J/(mol K)
E_A = 75_000        # J/mol, Literaturwert
T_versuch = 333.15  # K, entspricht 60 °C

# k_0 aus dem gemessenen k bei der Versuchstemperatur zurückrechnen
k_0 = k / np.exp(-E_A / (R * T_versuch))

print(f"k_0 = {k_0:.3e} 1/min")


def k_arrhenius(T, k_0, E_A, R=8.314):
    """Geschwindigkeitskonstante nach Arrhenius. T in K, E_A in J/mol."""
    return k_0 * np.exp(-E_A / (R * T))


for theta in [40, 60, 80, 100]:
    T = theta + 273.15
    k_T = k_arrhenius(T, k_0, E_A)
    print(f"{theta:4d} °C:  k = {k_T:.4f} 1/min,  t(90 %) = {zeit_fuer_umsatz(0.90, k_T):7.2f} min")

In [ ]:
plt.figure(figsize=(7, 4.5))

for theta in [40, 60, 80, 100]:
    T = theta + 273.15
    k_T = k_arrhenius(T, k_0, E_A)
    plt.plot(t_fein, konzentration_erster_ordnung(t_fein, 2.0, k_T),
             label=f"{theta} °C")

plt.plot(t, c_A, "ko", markersize=5, label="Messung bei 60 °C")

plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"Konzentration $c_\mathrm{A}$ / mol L$^{-1}$")
plt.title("Einfluss der Reaktionstemperatur")
plt.legend()
plt.grid(True)
plt.show()

Eine Temperaturerhöhung um 20 K verkürzt die Reaktionszeit hier um mehr als den Faktor 4 — die bekannte
Faustregel, dass sich die Reaktionsgeschwindigkeit je 10 K etwa verdoppelt, bestätigt sich für
diese Aktivierungsenergie.

> **Warnung vor Extrapolation:** Die Kurve für 100 °C ist eine reine Modellaussage. Ob die Reaktion
> dort noch erster Ordnung ist, ob Nebenreaktionen auftreten und ob das Lösungsmittel siedet, sagt
> Ihnen das Modell nicht. Extrapolierte Ergebnisse sind Hypothesen, keine Resultate.

---
## 4. Ausblick: wenn es keine Formel mehr gibt

Alles bisher beruhte darauf, dass die Differentialgleichung

$$\frac{\mathrm{d}c_\mathrm{A}}{\mathrm{d}t} = -k \, c_\mathrm{A}$$

eine geschlossene Lösung besitzt. Sobald die Aufgabe nur wenig realistischer wird, ist das vorbei:

- Folgereaktion $\mathrm{A} \to \mathrm{B} \to \mathrm{C}$ mit mehreren Stoffbilanzen,
- gemischte Reaktionsordnung, etwa $r = k\,c_\mathrm{A}^{1,5}$,
- gekoppelte Energiebilanz, weil die Reaktionswärme den Reaktor aufheizt und $k$ dadurch steigt.

Dann müssen Sie die Differentialgleichung **numerisch** lösen. Zur Einordnung — so sieht das aus:

In [ ]:
from scipy.integrate import solve_ivp

# Folgereaktion A -> B -> C, beide 1. Ordnung
k1, k2 = 0.15, 0.06     # 1/min


def bilanz(t, c):
    """Rechte Seite des Differentialgleichungssystems für A -> B -> C."""
    c_A, c_B, c_C = c
    return [-k1 * c_A,
            k1 * c_A - k2 * c_B,
            k2 * c_B]


loesung = solve_ivp(bilanz, [0, 60], [2.0, 0.0, 0.0], dense_output=True)

t_plot = np.linspace(0, 60, 300)
c_plot = loesung.sol(t_plot)

plt.figure(figsize=(7, 4.5))
plt.plot(t_plot, c_plot[0], label="A")
plt.plot(t_plot, c_plot[1], label="B (Zwischenprodukt)")
plt.plot(t_plot, c_plot[2], label="C")

plt.xlabel("Zeit $t$ / min")
plt.ylabel(r"Konzentration / mol L$^{-1}$")
plt.title("Folgereaktion A -> B -> C, numerisch gelöst")
plt.legend()
plt.grid(True)
plt.show()

i_max = np.argmax(c_plot[1])
print(f"Maximale Ausbeute an B: {c_plot[1][i_max]:.3f} mol/L nach {t_plot[i_max]:.1f} min")

Das Ergebnis ist verfahrenstechnisch hochrelevant: Das Zwischenprodukt B durchläuft ein Maximum.
Wollen Sie B gewinnen, müssen Sie die Reaktion zum richtigen Zeitpunkt abbrechen — eine
Optimierungsaufgabe, die es ohne numerische Lösung nicht zu beantworten gäbe.

Was `solve_ivp` intern tut, mit welchem Verfahren, welcher Schrittweite und welchem Fehler, ist
Gegenstand der **Numerik-Veranstaltung**. Sie haben mit diesen vier Notebooks das Handwerkszeug,
um dort einzusteigen.

---
## 5. Übungen

### Übung 4.1 — Ausreißer behandeln

Der Messwert bei t = 19 min fällt aus der Reihe (siehe Residuenplot). Wiederholen Sie die
Regression aus Abschnitt 3.2, aber nur mit den Messwerten bis einschließlich t = 15 min — mit der
Begründung, dass die Analytik darunter an ihre Nachweisgrenze kommt.

Wie stark ändern sich *k* und $R^2$? Formulieren Sie in einem Satz, ob die Änderung relevant ist.

In [ ]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
maske = t <= 15                  # ab 15 min nahe der Nachweisgrenze der HPLC
t_red, ln_c_red = t[maske], np.log(c_A[maske])

st, ab = np.polyfit(t_red, ln_c_red, deg=1)
k_red = -st

fit = st * t_red + ab
R2_red = 1 - np.sum((ln_c_red - fit)**2) / np.sum((ln_c_red - ln_c_red.mean())**2)

print(f"alle Werte:      k = {k:.4f} 1/min,  R^2 = {R2:.4f}")
print(f"nur t <= 15 min: k = {k_red:.4f} 1/min,  R^2 = {R2_red:.4f}")
print(f"relative Änderung von k: {(k_red - k)/k*100:+.2f} %")

# Die Änderung von k liegt im Bereich weniger Prozent und damit in der Größenordnung
# der Messunsicherheit - die Auswertung ist gegenüber dieser Entscheidung robust.
```

</details>

### Übung 4.2 — Reaktionsordnung prüfen

Wäre die Reaktion **zweiter** Ordnung, gälte

$$\frac{1}{c_\mathrm{A}} = \frac{1}{c_{\mathrm{A}0}} + k\,t$$

Trägt man also $1/c_\mathrm{A}$ über $t$ auf, müsste sich für zweite Ordnung eine Gerade ergeben.

Erstellen Sie beide Auftragungen nebeneinander (`plt.subplots`) und entscheiden Sie anhand von
$R^2$, welche Ordnung besser passt.

In [ ]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
def r_quadrat(x, y):
    """Bestimmtheitsmaß einer linearen Regression von y über x."""
    st, ab = np.polyfit(x, y, deg=1)
    fit = st * x + ab
    return 1 - np.sum((y - fit)**2) / np.sum((y - y.mean())**2), st, ab


R2_1, st1, ab1 = r_quadrat(t, np.log(c_A))
R2_2, st2, ab2 = r_quadrat(t, 1 / c_A)

print(f"1. Ordnung: R^2 = {R2_1:.4f}")
print(f"2. Ordnung: R^2 = {R2_2:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.plot(t, np.log(c_A), "o")
ax1.plot(t, st1 * t + ab1, "-")
ax1.set_xlabel("$t$ / min")
ax1.set_ylabel(r"$\ln c_\mathrm{A}$")
ax1.set_title(f"1. Ordnung, $R^2$ = {R2_1:.4f}")
ax1.grid(True)

ax2.plot(t, 1 / c_A, "o")
ax2.plot(t, st2 * t + ab2, "-")
ax2.set_xlabel("$t$ / min")
ax2.set_ylabel(r"$1/c_\mathrm{A}$ / L mol$^{-1}$")
ax2.set_title(f"2. Ordnung, $R^2$ = {R2_2:.4f}")
ax2.grid(True)

fig.tight_layout()
plt.show()

# Die Auftragung für 1. Ordnung liefert das deutlich höhere R^2 und zeigt keine
# systematische Krümmung. Die Reaktion ist erster Ordnung.
```

</details>

### Übung 4.3 — Auslegungsfrage

Ein Batch-Reaktor soll je Charge 500 L Produkt mit mindestens 95 % Umsatz liefern. Neben der
Reaktionszeit fallen pro Charge 45 min für Befüllen, Aufheizen, Entleeren und Reinigen an.

1. Schreiben Sie eine Funktion `chargenzeit(theta, X_ziel)`, die die gesamte Chargenzeit in
   Minuten liefert (Reaktionszeit nach Arrhenius plus 45 min Nebenzeit).
2. Berechnen Sie für 50, 60, 70, 80 und 90 °C die Chargenzeit und die Anzahl möglicher Chargen
   in einer 8-Stunden-Schicht.
3. Ab welcher Temperatur bringt eine weitere Erhöhung kaum noch Gewinn? Begründen Sie kurz.

In [ ]:
# Ihre Lösung:

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
NEBENZEIT = 45.0      # min pro Charge für Befüllen, Aufheizen, Entleeren, Reinigen
SCHICHT   = 8 * 60    # min


def chargenzeit(theta, X_ziel, k_0=k_0, E_A=E_A):
    """Gesamte Chargenzeit in min bei Temperatur theta in °C."""
    k_T = k_arrhenius(theta + 273.15, k_0, E_A)
    return zeit_fuer_umsatz(X_ziel, k_T) + NEBENZEIT


print(f"{'°C':>5s} {'t_Reaktion':>12s} {'t_Charge':>10s} {'Chargen/Schicht':>17s}")
print("-" * 47)

for theta in [50, 60, 70, 80, 90]:
    t_ges = chargenzeit(theta, 0.95)
    t_reak = t_ges - NEBENZEIT
    print(f"{theta:5d} {t_reak:12.1f} {t_ges:10.1f} {SCHICHT/t_ges:17.1f}")

# Ab etwa 80 °C wird die Chargenzeit von der konstanten Nebenzeit dominiert:
# Die Reaktionszeit ist dann klein gegenüber 45 min, weitere Temperaturerhöhung
# bringt kaum noch Durchsatz, erhöht aber Energiebedarf und Risiko von
# Nebenreaktionen. Das Optimum liegt hier bei 70 bis 80 °C.
```

</details>

### Übung 4.4 — Eigene Daten (freiwillig)

Nehmen Sie einen beliebigen Datensatz aus einem Ihrer Praktika — Aufheizkurve, Trocknungsverlauf,
Titrationskurve — speichern Sie ihn als CSV und werten Sie ihn nach demselben Schema aus:

1. Daten einlesen und ansehen
2. geeignete Linearisierung finden
3. Parameter durch Regression bestimmen
4. Modell und Messung vergleichen
5. Residuen prüfen

Dieses Schema deckt einen großen Teil der Auswertungen ab, die Ihnen im Studium begegnen werden.

---
## Zusammenfassung der Reihe

**Teil 1** — Variablen, Datentypen, f-Strings, Listen, Dictionaries
**Teil 2** — `if`/`for`/`while`, Funktionen, Docstrings, Fehlermeldungen lesen
**Teil 3** — NumPy-Arrays, Vektorisierung, lineare Gleichungssysteme, Matplotlib
**Teil 4** — Messdaten einlesen, Regression, vollständige Auswertung, gute Praxis

| Aufgabe | Befehl |
|---------|--------|
| CSV einlesen | `np.loadtxt(datei, delimiter=",", skiprows=6)` |
| Spalte herausziehen | `daten[:, 0]` |
| Ausgleichsgerade | `np.polyfit(x, y, deg=1)` |
| DGL numerisch lösen | `solve_ivp(f, [t0, t1], y0)` |

**Was Sie mitnehmen sollten, unabhängig von der Sprache:**

1. Rechnen Sie nichts, was Sie nicht vorher gezeichnet haben.
2. Jede Größe hat eine Einheit — Python weiß das nicht, Sie schon.
3. Jedes Ergebnis braucht eine unabhängige Kontrolle.
4. Wiederholung gehört in eine Funktion.
5. Ein Notebook, das nicht von oben durchläuft, ist nicht fertig.

Viel Erfolg — und weiter geht es in der Numerik.